# RUNNING THIS NOTEBOOK

## MODE 1 - Local (default, no setup needed)
Uses Weaviate Embedded + sentence-transformers. Runs on CPU. Free.

Run:
`jupyter notebook`

## MODE 2 - Production (Weaviate Cloud + OpenAI)
1. Create a free cluster at console.weaviate.cloud, then copy your cluster URL and API key.
2. Get an OpenAI API key at platform.openai.com.
3. Replace Cell 1 with:

```python
client = weaviate.connect_to_weaviate_cloud(
    cluster_url="YOUR_WEAVIATE_URL",
    auth_credentials=weaviate.auth.AuthApiKey("YOUR_WEAVIATE_API_KEY"),
    headers={"X-OpenAI-Api-Key": "YOUR_OPENAI_API_KEY"},
)
```

4. Replace collection creation in Cell 2 with:

```python
client.collections.create(
    name="EnterpriseKnowledgeBase",
    multi_tenancy_config=Configure.multi_tenancy(enabled=True),
    vectorizer_config=Configure.Vectorizer.text2vec_openai(),
    ...
)
```

5. Replace `tenant_search()` in Cell 5 with LangChain `WeaviateVectorStore` (see langchain-weaviate docs for a full example).

# Enterprise Multi-Tenant RAG with Weaviate + LangChain

This notebook demonstrates a **production-style multi-tenant RAG system** using Weaviate.

Key features:
- Tenant isolation
- Hybrid search (vector + BM25)
- Local embeddings (no API keys)
- LangChain-style retrieval

## 1. Setup Weaviate (Local Embedded) + Create Collection

In [1]:
import weaviate
from weaviate.embedded import EmbeddedOptions

client = weaviate.WeaviateClient(
    embedded_options=EmbeddedOptions()
)

client.connect()
print("Connected:", client.is_ready())

/Users/arkajamishra/Desktop/opensource_prs/recipes/integrations/llm-agent-frameworks/langchain/rag-with-multi-tenancy/.venv-run/lib/python3.13/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"warning","log_level_env":"","msg":"log level not recognized, defaulting to info","time":"2026-04-29T21:41:35+05:30"}
{"action":"startup","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"Feature flag LD integration disabled: could not locate WEAVIATE_LD_API_KEY env variable","time":"2026-04-29T21:41:35+05:30"}
{"action":"startup","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-04-29T21:41:35+05:30"}
{"action":"startup","auto_schema_enabled":{},"build_git_commit":"62dcafac32","build_go_version":

{"action":"read_disk_use","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"warning","msg":"disk usage currently at 80.57%, threshold set to 80.00%","path":"/Users/arkajamishra/.local/share/weaviate","time":"2026-04-29T21:41:35+05:30"}


{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"attempting to join","remoteNodes":{"Embedded_at_8079":"192.168.1.6:58211"},"time":"2026-04-29T21:41:36+05:30"}
{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"attempted to join and failed","remoteNode":"192.168.1.6:58211","status":8,"time":"2026-04-29T21:41:36+05:30"}
{"action":"raft","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","last-leader-addr":"","last-leader-id":"","level":"warning","msg":"heartbeat timeout reached, starting election","time":"2026-04-29T21:41:36+05:30"}
{"action":"raft","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"entering candidate state","node":{},"term":6,"time":"2026-04-29T21:4

{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"configured versions","server_version":"1.30.5","time":"2026-04-29T21:41:37+05:30","version":"1.30.5"}
{"action":"grpc_startup","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"grpc server listening at [::]:50060","time":"2026-04-29T21:41:37+05:30"}
{"action":"restapi_management","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"Serving weaviate at http://127.0.0.1:8079","time":"2026-04-29T21:41:37+05:30","version":"1.30.5"}
{"address":"192.168.1.6:58211","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"current Leader","time":"2026-04-29T21:41:37+05:30"}
{"build_git_commit":"62dcafac32","build

Connected: True


from weaviate.classes.config import Configure, Property, DataType

client.collections.delete("EnterpriseKnowledgeBase")

client.collections.create(
    name="EnterpriseKnowledgeBase",
    multi_tenancy_config=Configure.multi_tenancy(enabled=True),
    properties=[
        Property(name="content", data_type=DataType.TEXT),
        Property(name="source", data_type=DataType.TEXT),
        Property(name="department", data_type=DataType.TEXT),
    ]
)

print("Collection created")

## 2. Create Tenants (Isolation Layer)

In [2]:
from weaviate.classes.tenants import Tenant

collection = client.collections.get("EnterpriseKnowledgeBase")
existing = collection.tenants.get()

if not existing:
    collection.tenants.create([
        Tenant(name="engineering"),
        Tenant(name="finance"),
        Tenant(name="legal"),
    ])

print("Tenants ensured")

Tenants ensured


## 3. Local Embeddings (No API Key Needed)

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def embed(text):
    return model.encode(text).tolist()

{"action":"lsm_recover_from_active_wal_success","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","class":"EnterpriseKnowledgeBase","index":"enterpriseknowledgebase","level":"info","msg":"successfully recovered from write-ahead-log","path":"/Users/arkajamishra/.local/share/weaviate/enterpriseknowledgebase/engineering/lsm/property_source/segment-1777476288880931000.wal","shard":"engineering","time":"2026-04-29T21:41:37+05:30"}
{"action":"lsm_recover_from_active_wal_success","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","class":"EnterpriseKnowledgeBase","index":"enterpriseknowledgebase","level":"info","msg":"successfully recovered from write-ahead-log","path":"/Users/arkajamishra/.local/share/weaviate/enterpriseknowledgebase/engineering/lsm/objects/segment-1777476288880862000.wal","shard":"engineering","time":"2026-04-29T21:41:37+05:30"}
{"action":"lsm_rec

/Users/arkajamishra/Desktop/opensource_prs/recipes/integrations/llm-agent-frameworks/langchain/rag-with-multi-tenancy/.venv-run/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{"action":"lsm_recover_from_active_wal_success","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","class":"EnterpriseKnowledgeBase","index":"enterpriseknowledgebase","level":"info","msg":"successfully recovered from write-ahead-log","path":"/Users/arkajamishra/.local/share/weaviate/enterpriseknowledgebase/finance/lsm/property__id/segment-1777476289989109000.wal","shard":"finance","time":"2026-04-29T21:41:38+05:30"}
{"action":"lsm_recover_from_active_wal_success","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","class":"EnterpriseKnowledgeBase","index":"enterpriseknowledgebase","level":"info","msg":"successfully recovered from write-ahead-log","path":"/Users/arkajamishra/.local/share/weaviate/enterpriseknowledgebase/finance/lsm/property_content/segment-1777476289989993000.wal","shard":"finance","time":"2026-04-29T21:41:38+05:30"}
{"action":"lsm_recover_from_

{"action":"hnsw_prefill_cache_async","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"not waiting for vector cache prefill, running in background","time":"2026-04-29T21:41:39+05:30","wait_for_cache_prefill":false}
{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"Completed loading shard enterpriseknowledgebase_legal in 4.30325ms","time":"2026-04-29T21:41:39+05:30"}
{"action":"hnsw_vector_cache_prefill","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","count":3000,"index_id":"vectors_default","level":"info","limit":1000000000000,"msg":"prefilled vector cache","time":"2026-04-29T21:41:39+05:30","took":282958}


{"action":"read_disk_use","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"warning","msg":"disk usage currently at 80.56%, threshold set to 80.00%","path":"/Users/arkajamishra/.local/share/weaviate","time":"2026-04-29T21:42:06+05:30"}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8197.28it/s]

## 4. Ingest Documents per Tenant

Each tenant gets completely isolated data.

In [4]:
engineering_docs = [
    {"content": "RAG uses semantic chunking.", "source": "eng", "department": "engineering"},
    {"content": "Pipeline serves 5M predictions/day.", "source": "eng", "department": "engineering"},
]

finance_docs = [
    {"content": "Budget allocated $2.4M.", "source": "fin", "department": "finance"},
    {"content": "Cloud cost reduced 38%.", "source": "fin", "department": "finance"},
]

eng = collection.with_tenant("engineering")
fin = collection.with_tenant("finance")

with eng.batch.dynamic() as batch:
    for doc in engineering_docs:
        batch.add_object(
            properties=doc,
            vector=embed(doc["content"])
        )

with fin.batch.dynamic() as batch:
    for doc in finance_docs:
        batch.add_object(
            properties=doc,
            vector=embed(doc["content"])
        )

print("Data ingested")

Data ingested


## 5. Hybrid Search (Vector + Keyword)

This is critical for production RAG.

In [5]:
def tenant_search(tenant_name, query, top_k=2):
    col = client.collections.get("EnterpriseKnowledgeBase")
    tenant_col = col.with_tenant(tenant_name)

    query_vec = embed(query)

    results = tenant_col.query.hybrid(
        query=query,
        vector=query_vec,
        alpha=0.5,
        limit=top_k,
        return_properties=["content", "source", "department"]
    )

    if not results.objects:
        print(f"No results for tenant: {tenant_name}")
        return []

    return results.objects

results = tenant_search("engineering", "RAG system")
for r in results:
    print(r.properties["content"])

RAG uses semantic chunking.
RAG uses semantic chunking.


## 6. Metadata Filtering (Enterprise Control Layer)

In [6]:
from weaviate.classes.query import Filter

def filtered_search(tenant_name, query):
    col = client.collections.get("EnterpriseKnowledgeBase")
    tenant_col = col.with_tenant(tenant_name)

    results = tenant_col.query.hybrid(
        query=query,
        vector=embed(query),
        alpha=0.5,
        filters=Filter.by_property("department").equal(tenant_name),
        limit=2,
        return_properties=["content"]
    )

    return results.objects

results = filtered_search("engineering", "pipeline")
for r in results:
    print(r.properties["content"])

Pipeline serves 5M predictions/day.
Pipeline serves 5M predictions/day.


## 7. LangChain-style Retriever (Minimal)

In [7]:
from langchain_core.documents import Document

def langchain_retriever(tenant, query):
    results = tenant_search(tenant, query)

    return [
        Document(page_content=r.properties["content"], metadata=r.properties)
        for r in results
    ]

docs = langchain_retriever("engineering", "RAG")
for d in docs:
    print(d.page_content)

RAG uses semantic chunking.
RAG uses semantic chunking.
